In [ ]:
import pandas as pd
import re
import yaml

In [ ]:
import os
os.chdir('../../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

### Import data

In [ ]:
raw_text = pd.read_excel(dataset_config['path_other'] + 'EL_website_202409.xlsx')
raw_text

### (1) Extract firm name and alias

In [ ]:
# Extract firm name based on "Entity"
raw_text['firm_name'] = raw_text['Entity'].str.split(',').str[0]
raw_text

In [ ]:
# Extract alias based on "Entity"
pattern = r'—([^.,;]+)'
raw_text['alias'] = raw_text['Entity'].str.extractall(pattern)[0].groupby(level=0).apply('|'.join)
raw_text['alias'] = raw_text['alias'].fillna('')
raw_text

### （2）Extract time

In [ ]:
raw_text['date'] = raw_text['Federal Register citation'].str.extract(r'(\d{1,2}/\d{1,2}/\d{2})')
raw_text

In [ ]:
raw_text['date'] = raw_text['date'].apply(lambda x: x[:-2] + '20' + x[-2:] if pd.notnull(x) else x)
raw_text['date'] = raw_text['date'].apply(lambda x: '/'.join([x[-4:], x.split('/')[0], x.split('/')[1]]) if pd.notnull(x) else x)
raw_text


### (3) wide to long

In [ ]:
raw_text = raw_text.drop(columns=['Entity', 'Federal Register citation'])
raw_text

In [ ]:
# Define functions for data transformation
def reshape_data(df):
    result = []
    for _, row in df.iterrows():
        firm_name = row['firm_name']
        date = row['date']
        result.append([firm_name, date])  # Always include firm_name
        
        if row['alias']:  # If alias is not empty
            aliases = row['alias'].split('|')  # Split by '|'
            for alias in aliases:
                result.append([alias, date])
    
    # Creating a New DataFrame
    return pd.DataFrame(result, columns=['EL_name', 'date'])

# Applying data transformation functions
reshaped_data = reshape_data(raw_text)

In [ ]:
final = reshaped_data.drop_duplicates().dropna(subset="date")
final

### Export

In [ ]:
final.to_csv(dataset_config['path_processed'] + 'CN_CN/EL_website_cleaned.csv', index=False)